In [ ]:
import json
from openai import OpenAI
from config import NVIDIA_API_KEY, NVIDIA_BASE_URL,DEFAULT_MODEL

In [ ]:
client = OpenAI(
     base_url = NVIDIA_BASE_URL,
     api_key=NVIDIA_API_KEY
)

In [ ]:
SYSTEM_PROMPT = '''
You are an expert invoice processing agent for a business automation system.
Your job is to extract structured data from raw invoice text.

You MUST respond with ONLY valid JSON - no explanation, no markdow, no extra text. JUST follow the exact schema:

{
    "vendor_name":string, // Company that issued invoice.
    "invoice_number":string, // Invoice ID or reference number
    "invoice_date":string, // Formate YYYY-MM-DD
    "due_date":string, // Formate YYYY-MM-DD
    "line_items":[
        {
            "description":string,
            "quantity":number,
            "unit_price" number,
            "total":number,
        }
            ]
    "subtotal":number,
    "tax_amount":number,
    "total_amount":number,
    "currency":string,   // e.g. USD, EUR, INR
    "payment_status": string    // paid / unpaid / partial
}






'''

In [ ]:
sample_invoice='''
Invoice
From: TechSupplies India Pvt Ltd
Invoice No: INV-2025-00842
Date: 15th March 2025
Due Date: 15 April 2025

Bill To: Vikram's E-commerce Solution

Items:
-Wireless Mouse x 10 @ Rs 650 each = Rs. 6500
-USB-C hub X 5 @ Rs 1200 each = 6000
-HDMI Cable X 20 @ 300 each = 6000

Subtotal =  18500
GST (18%) = 3300
Total = 21800

Payment status Unpaid
'''

In [ ]:
sample_invoice='''
Invoice
From: TechSupplies India Pvt Ltd
Invoice No: INV-2025-00842
Date: 15th March 2025
Due Date: 15 April 2025

Bill To: Vikram's E-commerce Solution

Items:
-Wireless Mouse  @ Rs 650 each = Rs. 65000
-USB-C hub @ Rs 1200 each = 6000
-HDMI Cable @ 300 each = 6000

Subtotal =  77000
GST (18%) = 13860
Total = 90860

Payment status Unpaid
'''

In [ ]:
def extract_invoice_data(invoice:str)->dict:
    """
    Send invoice to NVIDIA LLM and returns structured JSON.
    
    '{
    "name":"",

    }'
    """
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages= [
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":invoice}
        ],
        temperature=0.05,#
        max_tokens=800
        )
    raw_output = response.choices[0].message.content
    invoice_data = json.loads(raw_output) #Dict
    return invoice_data


In [7]:
result = extract_invoice_data(sample_invoice)
print(result)

{'vendor_name': 'TechSupplies India Pvt Ltd', 'invoice_number': 'INV-2025-00842', 'invoice_date': '2025-03-15', 'due_date': '2025-04-15', 'line_items': [{'description': 'Wireless Mouse', 'quantity': 100, 'unit_price': 650, 'total': 65000}, {'description': 'USB-C hub', 'quantity': 5, 'unit_price': 1200, 'total': 6000}, {'description': 'HDMI Cable', 'quantity': 20, 'unit_price': 300, 'total': 6000}], 'subtotal': 77000, 'tax_amount': 13860, 'total_amount': 90860, 'currency': 'INR', 'payment_status': 'unpaid'}


In [8]:
sample_invoice

"\nInvoice\nFrom: TechSupplies India Pvt Ltd\nInvoice No: INV-2025-00842\nDate: 15th March 2025\nDue Date: 15 April 2025\n\nBill To: Vikram's E-commerce Solution\n\nItems:\n-Wireless Mouse  @ Rs 650 each = Rs. 65000\n-USB-C hub @ Rs 1200 each = 6000\n-HDMI Cable @ 300 each = 6000\n\nSubtotal =  77000\nGST (18%) = 13860\nTotal = 90860\n\nPayment status Unpaid\n"

In [9]:
import pandas as pd

In [10]:
from copy import deepcopy
import time
def process_response(response:dict, save_to_path:str=None):
    result_copy = deepcopy(response)
    payables_status = None
    inventory_data = pd.DataFrame(result_copy['line_items'])
    inventory_data['vendor_name'] = result_copy['vendor_name']
    inventory_data['invoice_number'] = result_copy['invoice_number']

    if result_copy['payment_status'] == "unpaid":
        del result_copy['line_items']
        payables = pd.DataFrame({k:[v] for k,v in zip(result_copy.keys(),result_copy.values())})
        payables_status = True
    if save_to_path:
        inventory_data.to_csv(f"Investory_data_{time.ctime().replace(":","_")}.csv", index=False)
        if payables_status:
            payables.to_csv(f"payable_data_{time.ctime().replace(":","_")}.csv", index=False)
            
    if payables_status:
        return [inventory_data,payables]
    return [inventory_data]


In [11]:
structured_response = process_response(response=extract_invoice_data(sample_invoice), save_to_path='.')

In [12]:
structured_response[1]

,vendor_name,invoice_number,invoice_date,due_date,subtotal,tax_amount,total_amount,currency,payment_status
0,TechSupplies India Pvt Ltd,INV-2025-00842,2025-03-15,2025-04-15,77000,13860,90860,INR,unpaid


In [13]:
import sqlite3
con = sqlite3.connect('E:\\Quant_fin\\1_Agentic_Ai\\SQL\\Agentic_Ai\\Agentic_ai.db')

structured_response[0].to_sql("inventory_table", con=con, if_exists="append", index=False)
structured_response[1].to_sql("invoice", con=con, if_exists="append", index=False)

1

In [14]:
bills = [
    """Supplier Name: ABC Traders
Invoice Number: INV001
Bill Date: 2026-04-10
Due Date: 2026-04-20

Items:
Item A | Qty: 10 | Price: 50 | Amount: 500
Item B | Qty: 5 | Price: 100 | Amount: 500

Sub Total: 1000
Tax: 180
Total: 1180""",

    """Supplier Name: XYZ Enterprises
Invoice Number: INV002
Bill Date: 2026-04-11
Due Date: 2026-04-25

Items:
Item C | Qty: 4 | Price: 200 | Amount: 800
Item D | Qty: 6 | Price: 100 | Amount: 600
Item E | Qty: 3 | Price: 200 | Amount: 600

Sub Total: 2000
Tax: 360
Total: 2360""",

    """Supplier Name: PQR Suppliers
Invoice Number: INV003
Bill Date: 2026-04-12
Due Date: 2026-04-30

Items:
Item F | Qty: 10 | Price: 50 | Amount: 500

Sub Total: 500
Tax: 90
Total: 590""",

    """Supplier Name: LMN Pvt Ltd
Invoice Number: INV004
Bill Date: 2026-04-08
Due Date: 2026-04-18

Items:
Item G | Qty: 5 | Price: 200 | Amount: 1000
Item H | Qty: 10 | Price: 50 | Amount: 500

Sub Total: 1500
Tax: 270
Total: 1770""",

    """Supplier Name: Global Mart
Invoice Number: INV005
Bill Date: 2026-04-05
Due Date: 2026-04-15

Items:
Item I | Qty: 10 | Price: 100 | Amount: 1000
Item J | Qty: 5 | Price: 200 | Amount: 1000
Item K | Qty: 10 | Price: 100 | Amount: 1000

Sub Total: 3000
Tax: 540
Total: 3540"""
]

In [15]:
for bill in bills:
    response_ = process_response(extract_invoice_data(bill))
    response_[0].to_sql("inventory_table", con=con, if_exists="append", index=False)
    response_[1].to_sql("invoice", con=con, if_exists="append", index=False)
    